# Week 66, Day 2: threading

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w066d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def worker(name, steps):
    for i in range(steps):
        yield f"{name}{i}"

threads = [worker("a", 3), worker("b", 2), worker("c", 1)]
log = []
while threads:
    for t in list(threads):
        try:
            log.append(next(t))
        except StopIteration:
            threads.remove(t)
print(log)

In [ ]:
import threading

def greet(name, times, out):
    for _ in range(times):
        out.append(f"hello {name} from {threading.current_thread().name}")

out = []
t = threading.Thread(target=greet, args=("Ada", 2, out), name="greeter")
print(t.name, t.is_alive(), t.daemon)
t.run()                                  # runs in this thread: no concurrency
print(out)
try:
    t.join()
except RuntimeError as e:
    print("RuntimeError:", e)

In [ ]:
import threading

def square_into(results, i, x):
    results[i] = x * x

values = [3, 4, 5]
results = [None] * len(values)
threads = [threading.Thread(target=square_into, args=(results, i, x)) for i, x in enumerate(values)]
for t in threads:
    t.run()                      # on your own computer: t.start(), then join each one
print(results)

def risky(report):
    try:
        1 / 0
    except Exception as e:
        report.append(type(e).__name__)     # pass the failure back explicitly

problems = []
threading.Thread(target=risky, args=(problems,)).run()
print(problems)

In [ ]:
import queue

jobs = queue.Queue(maxsize=3)
for n in (1, 2, 3):
    jobs.put(n)
try:
    jobs.put_nowait(4)
except queue.Full:
    print("full: a producer would wait here")

done = []
while True:
    try:
        n = jobs.get_nowait()
    except queue.Empty:
        break
    done.append(n * 10)
    jobs.task_done()
jobs.join()                      # returns at once: every item was marked done
print(done, jobs.empty())

pq = queue.PriorityQueue()
for item in [(2, "lunch"), (1, "fire"), (3, "email")]:
    pq.put(item)
print([pq.get()[1] for _ in range(3)])

In [ ]:
import threading

stop = threading.Event()
print(stop.is_set(), stop.wait(timeout=0))
stop.set()
print(stop.is_set(), stop.wait(timeout=0))

background = threading.Thread(target=lambda: None, daemon=True)
print(background.daemon)

data = threading.local()
data.user = "ada"                        # visible only in this thread
print(data.user, threading.current_thread().name)

In [ ]:
import heapq
from collections import deque

def run_threads(threads):
    "threads: name -> generator yielding None or ('sleep', t)."
    ready = deque(threads.items())
    sleepers, now, log, seq = [], 0, [], 0
    while ready or sleepers:
        if not ready:
            now, _, name, gen = heapq.heappop(sleepers)
            ready.append((name, gen))
            continue
        name, gen = ready.popleft()
        try:
            request = next(gen)
        except StopIteration:
            log.append((now, name, "finished"))
            continue
        if request and request[0] == "sleep":
            seq += 1
            heapq.heappush(sleepers, (now + request[1], seq, name, gen))
        else:
            ready.append((name, gen))
    return log

def downloader(seconds):
    yield ("sleep", seconds)              # waiting on the network

print(run_threads({"a": downloader(3), "b": downloader(1), "c": downloader(2)}))

> **Key idea.** threading.Thread(target=f, args=(x,)) creates a thread; start() runs it concurrently (once), join() waits for it, and run() just calls the target in the current thread. Targets' return values are discarded and their exceptions don't propagate: collect results in separate slots, a queue.Queue, or futures. queue.Queue is thread-safe: put, get (blocking), get_nowait, maxsize for back-pressure, task_done and join for accounting, and one sentinel per consumer to stop them. Event signals between threads (a stop flag); daemon threads are killed abruptly at exit, so stop real work with an Event and join; threading.local holds per-thread data. A generator with a scheduler simulates threads deterministically: each yield is a possible switch, and yielded requests can mean sleep or wait.

### Exercise: Taking turns

Write `round_robin(threads)`: run a dict of named generators one step at a time, in rotation (in the dict's order, skipping finished ones), until all are finished, and return a list of `(name, value)` pairs for every value yielded. Then `finish_order(threads)`: the names in the order the generators **finished**.

In [ ]:
from collections import deque

def round_robin(threads):
    ___

def finish_order(threads):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_roundrobin():\n    \"a round-robin scheduler\"\n    def counter(n):\n        for i in range(n):\n            yield i\n    assert round_robin({\"a\": counter(2), \"b\": counter(3), \"c\": counter(1)}) == [\n        (\"a\", 0), (\"b\", 0), (\"c\", 0), (\"a\", 1), (\"b\", 1), (\"b\", 2)]\n    assert finish_order({\"a\": counter(2), \"b\": counter(3), \"c\": counter(1)}) == [\"c\", \"a\", \"b\"]\n    assert round_robin({}) == [] and finish_order({\"x\": counter(0)}) == [\"x\"]")

### Exercise: Simulated time

Write `run_sleepers(threads)`: each generator yields a number of time units to **sleep** (0 means just let others run). Simulated time starts at 0. Ready threads run in the order they became ready; when none is ready, time jumps to the earliest wake-up (ties: the thread that went to sleep first). Return a dict from name to the time each thread finished. Then `timeline(threads)`: a list of `(time, name, value)` for every yield, in the order they happened.

In [ ]:
import heapq
from collections import deque

def run_sleepers(threads):
    ___

def timeline(threads):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sleepers():\n    \"sleeping threads\"\n    def job(*naps):\n        for n in naps:\n            yield n\n    assert run_sleepers({\"a\": job(3), \"b\": job(1, 1), \"c\": job(5)}) == {\"b\": 2, \"a\": 3, \"c\": 5}\n    assert timeline({\"a\": job(2, 2), \"b\": job(3)}) == [(0, \"a\", 2), (0, \"b\", 3), (2, \"a\", 2)]\n    assert run_sleepers({\"x\": job(0, 0), \"y\": job()}) == {\"y\": 0, \"x\": 0}\n    assert run_sleepers({\"p\": job(2), \"q\": job(2)}) == {\"p\": 2, \"q\": 2}")

### Exercise: Producers and consumers

Simulate a queue between generator threads. Write `SimQueue` with `put(item)` and a scheduler `run_pipeline(threads, queue)`: a thread yields `("put", item)` to add an item (then stays ready), or `("get",)` to take one; the scheduler **sends** the item into the generator (`gen.send(item)`) when one is available, and otherwise parks the thread until an item arrives (waiting threads are served in the order they started waiting). Threads run in turn from a FIFO ready queue. If threads are left waiting and none can run, raise `RuntimeError("deadlock")`. Return the list of thread names in the order they finished.

In [ ]:
from collections import deque

class SimQueue:
    def __init__(self):
        ___

def run_pipeline(threads, queue):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pipeline():\n    \"a producer-consumer simulation\"\n    q = SimQueue()\n    got = []\n    def producer(items):\n        for x in items:\n            yield (\"put\", x)\n        yield (\"put\", None)\n        yield (\"put\", None)\n    def consumer(name):\n        while True:\n            item = yield (\"get\",)\n            if item is None:\n                return\n            got.append((name, item))\n    order = run_pipeline({\"c1\": consumer(\"c1\"), \"c2\": consumer(\"c2\"), \"p\": producer([1, 2, 3])}, q)\n    assert sorted(x for _, x in got) == [1, 2, 3] and set(order) == {\"c1\", \"c2\", \"p\"}\n    assert got[0] == (\"c1\", 1) and got[1] == (\"c2\", 2)\n    q2 = SimQueue()\n    def needy():\n        yield (\"get\",)\n    try:\n        run_pipeline({\"n\": needy()}, q2)\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"deadlock\"")

### Exercise: The real queue.Queue

Using the real `queue.Queue` (which works without starting threads), write `fill(q, items)`: put every item, returning how many fitted before the queue was full (use `put_nowait` and stop at `queue.Full`); `drain(q)`: take every item with `get_nowait` until `queue.Empty`, returning them in order and calling `task_done()` for each; and `ordered_work(tasks)`: put `(priority, name)` pairs in a `queue.PriorityQueue` and return the names in the order they'd be processed.

In [ ]:
import queue

def fill(q, items):
    ___

def drain(q):
    ___

def ordered_work(tasks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_realqueue():\n    \"queue.Queue\"\n    import queue\n    q = queue.Queue(maxsize=3)\n    assert fill(q, [\"a\", \"b\", \"c\", \"d\"]) == 3 and q.full()\n    assert drain(q) == [\"a\", \"b\", \"c\"] and q.empty()\n    q.join()\n    unlimited = queue.Queue()\n    assert fill(unlimited, range(100)) == 100 and len(drain(unlimited)) == 100\n    assert ordered_work([(3, \"email\"), (1, \"fire\"), (2, \"lunch\")]) == [\"fire\", \"lunch\", \"email\"]")

### Exercise: Who finishes before exit?

A program's main thread runs for `main_time`; other threads are `(name, duration, daemon)` and all start at time 0. Python exits when the main thread **and** every non-daemon thread have finished; daemon threads still running then are killed. Write `exit_time(main_time, threads)` and `completed(main_time, threads)`: the sorted names of threads that finish (a daemon thread finishing exactly at the exit time counts as finished).

In [ ]:
def exit_time(main_time, threads):
    ___

def completed(main_time, threads):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_daemons():\n    \"daemon threads\"\n    threads = [(\"save\", 5, False), (\"heartbeat\", 100, True), (\"cache\", 4, True), (\"log\", 7, True)]\n    assert exit_time(2, threads) == 5 and completed(2, threads) == [\"cache\", \"save\"]\n    assert exit_time(10, threads) == 10 and completed(10, threads) == [\"cache\", \"log\", \"save\"]\n    assert exit_time(3, []) == 3 and completed(1, [(\"d\", 1, True)]) == [\"d\"]")

### Bug hunt: A string that should have been a tuple

`args=("Ada")` is just the string `"Ada"` in brackets, not a tuple, so the thread unpacks it into three arguments and calling it fails with `TypeError`. Add the comma that makes a one-item tuple so this prints `hello Ada`.

In [ ]:
import threading

def greet(name):
    print("hello", name)

t = threading.Thread(target=greet, args=("Ada"))
t.run()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"args is a tuple\"\n    expect_output(\"hello Ada\")")

### Bug hunt: A consumer that gives up too early

The consumer stops as soon as the queue is **momentarily** empty, but in this simulation the producer adds more items later, so the consumer quits early and this prints `[1]`. Have it stop only when it receives the sentinel `None`, so it prints `[1, 2, 3]`.

In [ ]:
from collections import deque

q = deque()
got = []

def producer():
    for x in (1, 2, 3):
        q.append(x)
        yield
        yield
    q.append(None)

def consumer():
    while True:
        if not q:
            return
        item = q.popleft()
        got.append(item)
        yield

threads = deque([producer(), consumer()])
while threads:
    t = threads.popleft()
    try:
        next(t)
        threads.append(t)
    except StopIteration:
        pass
print(got)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stop on the sentinel\"\n    expect_output(\"[1, 2, 3]\")")

### Bug hunt: Results that vanished

A thread's target return value is thrown away, so collecting `t.run()` results gives only `None`s and this prints `[None, None, None]`. Have each thread write its result into its own slot of a list made beforehand, so this prints `[1, 4, 9]`.

In [ ]:
import threading

def square(x):
    return x * x

results = []
for x in (1, 2, 3):
    t = threading.Thread(target=square, args=(x,))
    results.append(t.run())
print(results)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"results in slots\"\n    expect_output(\"[1, 4, 9]\")")

### Puzzle: Starting and joining

Arrange the lines that start several threads so they run at the same time, then wait for all of them (as you'd write it on your own computer). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # what Thread.start() returns
q = ___   # True or False: a thread's target's return value is available from t.join()
r = ___   # the queue method that raises queue.Empty instead of waiting
s = ___   # True or False: daemon threads run their finally blocks when the program exits
t = ___   # how many sentinels to stop 4 consumer threads

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is None and q is False and r == \"get_nowait\", \"p, q and r\"\n    assert s is False and t == 4, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a cooperative threading library

Write a scheduler for generator threads that supports the main things real threads do. `Scheduler()` has `spawn(name, gen)` and `run()`, which runs everything and returns a list of `(time, name, message)` log entries. A thread can yield these requests:

- `("log", message)`: record a log entry at the current time, then carry on;
- `("sleep", t)`: wake after `t` units of simulated time;
- `("join", name)`: wait until the named thread has finished (at once if it already has);
- `("spawn", name, gen)`: start a new thread (added to the end of the ready queue);
- `("wait", event)` and `("set", event)`: events are strings; `wait` blocks until the event has been set (at once if it already has been), and `set` wakes every waiting thread.

Ready threads run in FIFO order; a woken thread joins the back of the ready queue. When nothing is ready, time jumps to the next wake-up. If threads are still blocked when nothing can run, `run()` raises `RuntimeError` naming the blocked threads, sorted: `"deadlock: a, b"`. Also give the scheduler `finished`, a dict from name to finishing time.

In [ ]:
import heapq
from collections import deque

class Scheduler:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_scheduler():\n    \"spawning, sleeping, joining and events\"\n    s = Scheduler()\n    def download(n):\n        yield (\"log\", f\"start {n}\")\n        yield (\"sleep\", n)\n        yield (\"log\", f\"done {n}\")\n    def main():\n        yield (\"spawn\", \"d3\", download(3))\n        yield (\"spawn\", \"d1\", download(1))\n        yield (\"join\", \"d3\")\n        yield (\"log\", \"all downloaded\")\n        yield (\"set\", \"ready\")\n    def listener():\n        yield (\"wait\", \"ready\")\n        yield (\"log\", \"heard ready\")\n    s.spawn(\"main\", main())\n    s.spawn(\"listener\", listener())\n    log = s.run()\n    assert log == [(0, \"d3\", \"start 3\"), (0, \"d1\", \"start 1\"), (1, \"d1\", \"done 1\"), (3, \"d3\", \"done 3\"),\n                   (3, \"main\", \"all downloaded\"), (3, \"listener\", \"heard ready\")]\n    assert s.finished == {\"d1\": 1, \"d3\": 3, \"main\": 3, \"listener\": 3}\n\ndef test_deadlock():\n    \"blocked threads\"\n    s = Scheduler()\n    def a():\n        yield (\"join\", \"b\")\n    def b():\n        yield (\"wait\", \"never\")\n    s.spawn(\"a\", a())\n    s.spawn(\"b\", b())\n    try:\n        s.run()\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"deadlock: a, b\"\n    s2 = Scheduler()\n    def early():\n        yield (\"set\", \"go\")\n    def late():\n        yield (\"sleep\", 2)\n        yield (\"wait\", \"go\")\n        yield (\"join\", \"early\")\n        yield (\"log\", \"fine\")\n    s2.spawn(\"early\", early())\n    s2.spawn(\"late\", late())\n    assert s2.run() == [(2, \"late\", \"fine\")]")

## Warm-up

Three generators took turns, one step each, until all were finished. That's a **scheduler**: real threads interleave in a similar way, except that the operating system decides when to switch, at almost any point, and the order can change from run to run.

## Thread basics

```
import threading

def greet(name, times):
    for _ in range(times):
        print("hello", name)

t = threading.Thread(target=greet, args=("Ada", 2), name="greeter")
t.start()        # begins running greet in a new thread; returns at once
t.join()         # waits until the thread has finished
```

- `Thread(target=func, args=(...), kwargs={...}, name=..., daemon=...)` creates a thread object; nothing runs until **`start()`**. `args` must be a **tuple**: `args=("Ada",)` with the comma.
- `start()` may only be called once per thread; `join(timeout=None)` waits for it (it's an error to join a thread that hasn't started); `is_alive()` says whether it's still running.
- `threading.current_thread().name`, `threading.active_count()` and `threading.enumerate()` describe the running threads.
- Calling `t.run()` directly runs the target **in the current thread**, like an ordinary call: occasionally useful in tests, but it isn't concurrency.
- You can also subclass `Thread` and override `run()`, though passing a `target` is usually simpler.

In the browser, `Thread.start()` raises `RuntimeError` (it can't start new threads), but you can still create thread objects and call `run()`:

## Getting results back

`Thread` has no return value: whatever the target returns is thrown away. Ways to get results out:

- **A shared container** that each thread writes to its own slot: `results[i] = value` in a list made beforehand, or a dict with distinct keys per thread. Appending to one shared list from many threads happens to be safe in CPython (each `append` is atomic), but relying on that is fragile; separate slots are clearer.
- **A `queue.Queue`**: each thread puts its result, and the main thread gets them. Thread-safe by design.
- **`concurrent.futures`** (Friday): `executor.submit(func, x)` returns a `Future` whose `.result()` is the return value, and exceptions come back too. Usually the best choice.

Exceptions in a thread don't reach the thread that started it: they're printed (through `threading.excepthook`) and the thread ends. Catch them inside the target and report them, or use futures.

## Queues: producers and consumers

The cleanest way for threads to cooperate is not to share data structures at all, but to **pass messages** through a `queue.Queue` (thread-safe; all locking is inside it):

- `q.put(item)` adds an item (waiting if the queue has a `maxsize` and is full); `q.get()` removes the oldest, **waiting** while the queue is empty; `get_nowait()` / `put_nowait()` raise `queue.Empty` / `queue.Full` instead of waiting; `get(timeout=1)` waits at most that long.
- **Work accounting**: after processing an item, a consumer calls `q.task_done()`; `q.join()` waits until every item put has been marked done.
- `queue.LifoQueue` (a stack) and `queue.PriorityQueue` (smallest first) have the same interface.

The **producer-consumer** pattern: producers put work on a queue, a fixed number of consumer threads take items and process them. To stop the consumers, put one **sentinel** value (often `None`) per consumer; each consumer stops when it gets one. A `maxsize` gives **back-pressure**: fast producers wait instead of filling memory.

## Events, daemons and thread-local data

- **`threading.Event`**: a flag threads can wait for. `event.set()` raises it, `event.clear()` lowers it, `event.wait(timeout=None)` blocks until it's set, and `event.is_set()` checks without waiting. A common use is a **stop signal**: worker loops check `while not stop.is_set():`, and the main thread calls `stop.set()` to ask them to finish.
- **Daemon threads** (`daemon=True`) don't keep the program alive: when the last non-daemon thread (usually the main thread) finishes, the program exits and daemon threads are stopped abruptly, without running `finally` blocks or closing files. Use them only for work that's safe to abandon (a heartbeat, a cache refresher); for anything else, stop threads with an `Event` and `join` them.
- **`threading.local()`**: an object whose attributes are separate for each thread: each thread sees its own `local.value`. Useful for per-thread resources such as a database connection.

## A simulator you can run

To experiment here, model each thread as a **generator**: its code runs until a `yield`, which hands control back to the scheduler. The `yield` marks the points where a real thread might be paused. What a thread yields can be a **request** to the scheduler:

- `yield` (nothing): "I'm happy to let others run now";
- `yield ("sleep", t)`: "wake me after `t` units of simulated time";
- `yield ("get", queue)`: "give me an item from this queue, waiting while it's empty".

The scheduler keeps a **ready queue** of threads that can run, a heap of **sleepers** ordered by wake-up time, and lists of threads **waiting** for something. It repeatedly runs the next ready thread until its next `yield`. Because the generator suspends exactly where it yielded, its local variables survive, just as a real thread's stack does. This is also how `asyncio` works (next week), with `await` in place of `yield`.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, change the "On your own computer" downloader from yesterday to use a `queue.Queue` and three consumer threads with sentinels. Tomorrow: the global interpreter lock, and why threads don't make CPU-bound Python code faster.